In [1]:
import datetime
import os

import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl

from tqdm.auto import tqdm
from src.thetadata_pipeline.loaders.tick import ensure_option_quote_windows, ensure_stock_tick_windows, StockTickWindow
from src.thetadata_pipeline.pipeline_config import load_pipeline_config
from src.thetadata_pipeline.settings import get_settings
from src.thetadata_pipeline.time_utils import to_date, ms_of_day

cfg = load_pipeline_config()
settings = get_settings()

DATE_CUT = datetime.date(2021, 1, 1)
TICKER: str = "SPY"
NEED_TIME: float = 9.6
CUT_DELTA_UP: float = 0.40
CUT_DELTA_DONW: float = 0.10
NEED_DELTA: float = 0.35

# Trades

In [2]:
need_time = int(NEED_TIME * 1_000 * 60 * 60)
options_df = pl.DataFrame()
for file in settings.m1_with_greeks_dir.glob(f'*{TICKER}_m1_greeks_opts.parquet'):
    cur_df = (
        pl.read_parquet(file).filter(
            (pl.col('ms_of_day') >= need_time) &
            (
                ((pl.col('delta_bid') <= CUT_DELTA_UP) & (pl.col('delta_bid') >= CUT_DELTA_DONW) & (pl.col('right') == 'c')) |
                ((pl.col('delta_bid') >= -CUT_DELTA_UP) & (pl.col('delta_bid') <= -CUT_DELTA_DONW) & (pl.col('right') == 'p'))
            )
        )
    )
    options_df = pl.concat([options_df, cur_df], how='diagonal')

len(options_df['date'].unique())

1331

In [3]:
options_df = options_df.filter(pl.col('date') > DATE_CUT)

In [4]:
options_df = options_df.with_columns(
        spread=(pl.col('ask') - pl.col('bid')) / pl.col('ask')
    ).filter(pl.col('spread') < 0.1)

start_df = options_df\
    .sort('date', 'ms_of_day')\
    .unique(['date', 'ms_of_day', 'right'], keep='first', maintain_order=True)\
    .group_by(['date', 'ms_of_day']).agg(pl.col('ticker').count())\
    .sort('date', 'ms_of_day')\
    .filter(pl.col('ticker') > 1)\
    .unique('date', keep='first', maintain_order=True)\
    .rename({'ms_of_day': 'start_time'})\
    .drop('ticker')
print(start_df.shape)

options_df = options_df.join(
        start_df,
        on='date',
        how='inner'
    ).filter(pl.col('ms_of_day') >= pl.col('start_time'))\
    .drop('spread', 'start_time')

(1172, 2)


In [5]:
trades_dict = {}
trades_dict['call'] = options_df.filter(pl.col('right') == 'c')\
    .with_columns(delta_diff=(pl.col('delta_bid') - NEED_DELTA).abs())\
    .sort(['date', 'expiration', 'ms_of_day', 'delta_diff'])\
    .unique(subset=['date', 'expiration'], keep='first', maintain_order=True)\
    .drop([
        'bid_size', 'bid_exchange', 'bid_condition', 'ask_size', 'ask_exchange', 'ask_condition',
        'dgs1', 'timeToExp', 'gamma_ask', 'theta_ask', 'vega_ask', 'rho_ask', 'IV_ask', 'delta_ask',
        'gamma_bid', 'theta_bid', 'vega_bid', 'rho_bid', 'delta_diff'
    ])

trades_dict['put'] = options_df.filter(pl.col('right') == 'p')\
    .with_columns(delta_diff=(pl.col('delta_bid') + NEED_DELTA).abs())\
    .sort(['date', 'expiration', 'ms_of_day', 'delta_diff'])\
    .unique(subset=['date', 'expiration'], keep='first', maintain_order=True)\
    .drop([
        'bid_size', 'bid_exchange', 'bid_condition', 'ask_size', 'ask_exchange', 'ask_condition',
        'dgs1', 'timeToExp', 'gamma_ask', 'theta_ask', 'vega_ask', 'rho_ask', 'IV_ask', 'delta_ask',
        'gamma_bid', 'theta_bid', 'vega_bid', 'rho_bid', 'delta_diff'
    ])

len(trades_dict['call']), len(trades_dict['put'])

(1172, 1172)

In [6]:
cur_df = trades_dict['call'][['date', 'IV_bid']].join(
        trades_dict['put'][['date', 'IV_bid']].rename({'IV_bid': 'IV_bid_put'}),
        on='date'
    ).with_columns(avgIV=(pl.col('IV_bid') + pl.col('IV_bid_put')) / 2)\
    [['date', 'avgIV']].to_pandas()\
    .set_index('date')\
    .rolling(window='365D')\
    .mean()

px.line(cur_df, title='Вся динамика капитала зависит от волатильности. Чем она выше, тем выше профит. Раньше эта стратегия была не доступна из-за спредов и малого количество экпираций в неделе.')

# Preparing for getting trade data

In [7]:
ticker_m1 = pl.DataFrame()
for file in settings.stock_m1_dir.glob(f'*{TICKER}_m1_stock.parquet'):
    ticker_m1 = pl.concat([ticker_m1, pl.read_parquet(file)])
ticker_m1 = (
    ticker_m1.with_columns(pl.col('date').cast(pl.String).str.to_datetime('%Y%m%d').cast(pl.Date))
    .filter((pl.col('date') > DATE_CUT) & (pl.col('close') > 0))
    .sort('date', 'ms_of_day')
)

In [8]:
def latency_distribution(latency_col: pl.Series | pd.Series, lengh: int):
    percentiles = np.linspace(0, 100, lengh)
    new_list = np.round(np.percentile(latency_col * 1000, percentiles), -2)
    new_list = np.maximum(new_list, 0)
    np.random.shuffle(new_list)

    return new_list


latten_path = settings.ib_states_dir / 'U1717377_lattency.csv'
lattency = pl.read_csv(latten_path).filter((pl.col('lattency') < 120))

open_lattency = lattency.filter(pl.col('lable') == 'open')['lattency']
stop_lattency = lattency.filter(pl.col('lable') == 'stop')['lattency']

# Enter / Exit prices

In [9]:
for right in list(trades_dict.keys()):
    print(f"---------{right}---------")
    trade_df = trades_dict[right]

    # Enter Prices------
    trade_df = (
        trade_df
        .with_columns(start_ms=pl.col('ms_of_day') + latency_distribution(open_lattency, trade_df.height))
        .with_columns(end_ms=pl.col('start_ms'))
    )

    print(f"{len(trade_df)} {right} - enter trades")
    answ = (
        ensure_option_quote_windows(settings, trade_df, option_concurrency=8)
        .with_columns(pl.col('right').str.to_lowercase().alias('right'))
        .rename({'bid': 'ent_opt_bid', 'ask': 'ent_opt_ask', 'time': 'ent_time'})
        ['date', 'expiration', 'strike', 'right', 'ent_time', 'ent_opt_bid', 'ent_opt_ask']
    )

    trade_df = (
        trade_df.join(answ, on=['date', 'expiration', 'strike', 'right'], how='inner')
        .drop(['end_ms', 'bid', 'ask', 'ms_of_day'])
        .rename({'start_ms': 'ent_time_ms'})
    )

    # Exit Prices ------
    # M1 crosses
    total_m1 = pl.DataFrame()
    for row in trade_df.iter_rows(named=True):
        need_date = row['date']
        need_time = row['ent_time_ms']

        if right == 'call':
            cur_m1 = ticker_m1.filter(
                    (need_date == pl.col('date')) & (need_time < pl.col('ms_of_day')) &
                    (row['strike'] < pl.col('high'))
                )
        elif right == 'put':
            cur_m1 = ticker_m1.filter(
                    (need_date == pl.col('date')) & (need_time < pl.col('ms_of_day')) &
                    (row['strike'] > pl.col('low'))
                )
        else:
            raise ValueError(f"Wrong right: {right}")

        if len(cur_m1) == 0:
            continue

        cur_m1 = cur_m1[0]['ms_of_day', 'high', 'date'].rename({'date': 'date_exit'})\
            .with_columns(
                expiration=pl.lit(row['expiration']),
                strike=pl.lit(row['strike']),
                date=pl.lit(row['date']),
                right=pl.lit(row['right']),
                ticker=pl.lit(row['ticker']),
            )
        total_m1 = pl.concat([total_m1, cur_m1])

    total_m1 = (
        total_m1.with_row_index(name='idx')
        .rename({'ms_of_day': 'start_ms'})
        .with_columns(end_ms=pl.col('start_ms') + 300_000)
    )
    print(f"{len(total_m1)} {right} - stop trades")

    # Crosses from ticks
    tick_df = (
        ensure_stock_tick_windows(settings, total_m1, concurrency=8, interval='tick')
        .rename({'time': 'exact_stop_time', 'bid': 'quote_bid', 'ask': 'quote_ask'})
    )

    if right == 'call':
        total_m1_with_ticks = total_m1.join(tick_df, on='date', how='left').filter(pl.col('quote_ask') > pl.col('strike'))
    elif right == 'put':
        total_m1_with_ticks = total_m1.join(tick_df, on='date', how='left').filter(pl.col('quote_bid') < pl.col('strike'))
    else:
        raise ValueError(f"Wrong right: {right}")

    total_m1_with_ticks = (
        total_m1_with_ticks.sort('idx', 'date', 'exact_stop_time')
        .unique('idx', keep="first", maintain_order=True)
    )

    idx_dif = set(total_m1['idx']) - set(total_m1_with_ticks['idx'])
    empty_data = total_m1.filter(pl.col('idx').is_in(idx_dif))

    print(f"{len(empty_data)} {right} - empty ticks")
    empty_data = (
        empty_data.join(tick_df, on='date', how='left')
        .with_columns(
            diff=pl.when(right == 'call')
            .then((pl.col('strike') - pl.col('quote_bid')).abs())
            .otherwise((pl.col('strike') - pl.col('quote_ask')).abs())
        ).sort('idx', 'diff')
        .unique('idx', keep="first", maintain_order=True)
        .drop('diff')
    )
    total_m1 = (
        pl.concat([total_m1_with_ticks, empty_data])
        .sort('idx')
        .drop(['quote_bid', 'quote_ask', 'start_ms', 'end_ms'])
    )
    total_m1 = (
        total_m1.with_columns(ms_of_day(total_m1['exact_stop_time']).alias('exact_stop_time_ms'))
        .with_columns(start_ms=pl.col('exact_stop_time_ms') + latency_distribution(stop_lattency, total_m1.height))
        .with_columns(end_ms=pl.col('start_ms'))
    )

    # Options' exit prices
    print(f"{len(total_m1)} {right} - exit trades")
    answ = ensure_option_quote_windows(settings, total_m1, option_concurrency=8)
    opt_ms = (
        answ.with_columns(
            pl.col('right').str.to_lowercase().alias('right'),
            ms_of_day(answ['time']).cast(pl.Float64).alias('time_ms')
        ).rename({'bid': 'ext_opt_bid', 'ask': 'ext_opt_ask'})
        ['date', 'expiration', 'strike', 'right', 'time', 'time_ms', 'ext_opt_bid', 'ext_opt_ask']
    )

    stop_trades = total_m1.height
    total_m1 = (
        total_m1.join(
            opt_ms,
            left_on=['date_exit', 'expiration', 'strike', 'right'],
            right_on=['date', 'expiration', 'strike', 'right'],
            how='inner'
        ).filter(pl.col('time_ms') >= pl.col('start_ms'))
        .sort('idx', 'date', 'time_ms')
        .unique('idx', keep="first", maintain_order=True)
    )
    assert total_m1.height == stop_trades, f"Wrong length of total_m1: {len(total_m1)} != {stop_trades}"
    total_m1 = total_m1.rename({'time': 'ext_time', 'start_ms': 'ext_time_ms'}).drop('end_ms')

    # Final join
    trades_dict[f'{right}_final'] = trade_df.join(
            total_m1.drop('high', 'date_exit'),
            on=['expiration', 'strike', 'right', 'date', 'ticker'],
            how='left'
        ).with_columns(
            ext_opt_bid=pl.col('ext_opt_bid').fill_null(0.02),
            ext_opt_ask=pl.col('ext_opt_ask').fill_null(0.02)
        ).with_columns(
            (pl.col('ent_opt_bid') - pl.col('ext_opt_ask')).alias(f'{right}_profit')
        ).drop(['idx'])

---------call---------
1172 call - enter trades
Option quotes 100ms: windows=1172, theta_ranges=448, concurrency=8, tickers=SPY
SPY. Option quotes cache: H:\Market\ThetaData\data\options\100ms\SPY_100ms_opts.parquet
Option quotes downloading ranges=448, concurrency=8


Option quotes download: 100%|██████████| 448/448 [02:51<00:00,  2.61range/s, date=2026-05-14, end=34573600, rows=1, start=34573600, ticker=SPY]


SPY. Option quotes cache merge: staged_files=448
745 call - stop trades
Stock ticks. requested_ranges=745, theta_ranges=0
6 call - empty ticks
745 call - exit trades
Option quotes 100ms: windows=745, theta_ranges=365, concurrency=8, tickers=SPY
SPY. Option quotes cache: H:\Market\ThetaData\data\options\100ms\SPY_100ms_opts.parquet
Option quotes downloading ranges=365, concurrency=8


Option quotes download: 100%|██████████| 365/365 [02:44<00:00,  2.22range/s, date=2026-05-15, end=37124587, rows=1, start=37124587, ticker=SPY]


SPY. Option quotes cache merge: staged_files=365
---------put---------
1172 put - enter trades
Option quotes 100ms: windows=1172, theta_ranges=627, concurrency=8, tickers=SPY
SPY. Option quotes cache: H:\Market\ThetaData\data\options\100ms\SPY_100ms_opts.parquet
Option quotes downloading ranges=627, concurrency=8


Option quotes download: 100%|██████████| 627/627 [03:25<00:00,  3.05range/s, date=2026-05-12, end=34567000, rows=1, start=34567000, ticker=SPY]


SPY. Option quotes cache merge: staged_files=627
718 put - stop trades
Stock ticks. requested_ranges=718, theta_ranges=0
4 put - empty ticks
718 put - exit trades
Option quotes 100ms: windows=718, theta_ranges=368, concurrency=8, tickers=SPY
SPY. Option quotes cache: H:\Market\ThetaData\data\options\100ms\SPY_100ms_opts.parquet
Option quotes downloading ranges=368, concurrency=8


Option quotes download: 100%|██████████| 368/368 [02:23<00:00,  2.56range/s, date=2026-04-21, end=39501661, rows=1, start=39501661, ticker=SPY]


SPY. Option quotes cache merge: staged_files=368


In [11]:
strangle_trades = trades_dict['call_final'].join(
        trades_dict['put_final'],
        on=['expiration', 'date', 'ticker', 'baseClose'],
        how='inner',
        suffix='_put'
    )
assert len(trades_dict['call_final']) == len(trades_dict['put_final']) == len(strangle_trades)

strangle_trades = strangle_trades.with_columns(
        strangle_profit=pl.col('call_profit') + pl.col('put_profit'),
        avgIV=(pl.col('IV_bid') + pl.col('IV_bid_put')) / 2
    )